## Task 2 ##

In [7]:
import cv2
import numpy as np
import math

# Initialize background subtractor (MOG2 algorithm)
bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=False)

# Define exit zone parameters
EXIT_ZONE_PERCENT_HEIGHT = 0.48  # bottom 48%
EXIT_ZONE_PERCENT_WIDTH = 0.04   # left 4%
MOTION_THRESHOLD = 3  # Vehicle must move left for at least 3 consecutive frames before counting

class CentroidTracker:
    def __init__(self, dist_threshold=25):
        self.objects = {} 
        self.next_id = 0
        self.dist_threshold = dist_threshold
        self.initial_x = {} 
        self.counted_ids = set() 
        self.motion_history = {} 
    
    def update(self, detections):
        objects_with_id = []
        new_centroids = []
        for (x, y, w, h) in detections:
            cx = x + w//2
            cy = y + h//2
            new_centroids.append((cx, cy, x, y, w, h))
        
        updated_ids = {}

        for (cx, cy, x, y, w, h) in new_centroids:
            match_id = None
            min_dist = self.dist_threshold
            for obj_id, (prev_cx, prev_cy) in self.objects.items():
                dist = math.hypot(cx - prev_cx, cy - prev_cy)
                if dist < min_dist:
                    min_dist = dist
                    match_id = obj_id
            
            if match_id is not None:
                # Update existing object
                self.objects[match_id] = (cx, cy)
                updated_ids[match_id] = True
                objects_with_id.append([x, y, w, h, match_id])

                # Store recent x-positions to check consistent leftward motion
                if match_id not in self.motion_history:
                    self.motion_history[match_id] = []
                self.motion_history[match_id].append(cx)

                # Keep only the last motion threshold positions
                if len(self.motion_history[match_id]) > MOTION_THRESHOLD:
                    self.motion_history[match_id].pop(0)

            else:
                # Register new object
                obj_id = self.next_id
                self.objects[obj_id] = (cx, cy)
                self.initial_x[obj_id] = cx
                objects_with_id.append([x, y, w, h, obj_id])
                self.next_id += 1

                # Initialize motion history
                self.motion_history[obj_id] = [cx]
        
        current_ids = set([obj[-1] for obj in objects_with_id])
        disappeared_ids = [obj_id for obj_id in list(self.objects.keys()) if obj_id not in current_ids]

        for obj_id in disappeared_ids:
            self.objects.pop(obj_id, None)
            self.initial_x.pop(obj_id, None)
            self.motion_history.pop(obj_id, None)  # Remove motion history when object disappears
        
        return objects_with_id

def count_vehicles(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Unable to open video: {video_path}")
        return 0
    
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)  # Get video frame rate
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # Total frames in video
    total_seconds = total_frames / fps  # Total duration of video in seconds
    total_minutes = total_seconds / 60  # Total duration in minutes

    exit_zone_x = int(EXIT_ZONE_PERCENT_WIDTH * frame_width)
    exit_zone_y = int((1.0 - EXIT_ZONE_PERCENT_HEIGHT) * frame_height)
    
    tracker = CentroidTracker(dist_threshold=30)
    total_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        fg_mask = bg_subtractor.apply(gray)
        
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))
        mask_clean = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel)
        mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_OPEN, kernel)
        mask_clean = cv2.dilate(mask_clean, kernel, iterations=2)
        
        _, mask_clean = cv2.threshold(mask_clean, 127, 255, cv2.THRESH_BINARY)
        
        contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        detections = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 500:
                continue
            x, y, w, h = cv2.boundingRect(cnt)
            detections.append((x, y, w, h))
        
        objects = tracker.update(detections)

        for (x, y, w, h, obj_id) in objects:
            cx = x + w//2
            cy = y + h//2
            if obj_id not in tracker.counted_ids:
                # Check if object is in exit zone and moving left for motion threshold frames
                if (cx < exit_zone_x and cy > exit_zone_y and tracker.initial_x.get(obj_id, frame_width) > exit_zone_x):
                    x_positions = tracker.motion_history.get(obj_id, [])
                    if len(x_positions) >= MOTION_THRESHOLD and all(x_positions[i] > x_positions[i + 1] for i in range(len(x_positions) - 1)):
                        total_count += 1
                        tracker.counted_ids.add(obj_id)

        cv2.rectangle(frame, (0, exit_zone_y), (exit_zone_x, frame_height), (0, 0, 255), 2)
        cv2.putText(frame, "Exit Zone", (10, exit_zone_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        cv2.putText(frame, f"Count: {total_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        
        cv2.imshow("Vehicle Counter", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    
    avg_cars_per_min = total_count / total_minutes if total_minutes > 0 else 0
    print(f"Total vehicles counted: {total_count}")
    print(f"Total video duration: {total_minutes:.2f} minutes")
    print(f"Average vehicles per minute: {avg_cars_per_min:.2f}")
    
    return total_count

# Run the function on video files
count1 = count_vehicles('Traffic_Laramie_1.mp4')
print("Vehicles exited left (Traffic_Laramie_1):", count1)

count2 = count_vehicles('Traffic_Laramie_2.mp4')
print("Vehicles exited left (Traffic_Laramie_2):", count2)

Total vehicles counted: 6
Total video duration: 2.97 minutes
Average vehicles per minute: 2.02
Vehicles exited left (Traffic_Laramie_1): 6
Total vehicles counted: 5
Total video duration: 1.76 minutes
Average vehicles per minute: 2.84
Vehicles exited left (Traffic_Laramie_2): 5
